In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import sys

sys.path.append("..")

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [3]:
CKP = "/system/user/publicwork/gyrokinetics_checkpoints/good_one"
device = "cuda"

In [ ]:
import yaml
from omegaconf import OmegaConf

from utils import load_model_and_config
from models import get_model

from dataset.cyclone import CycloneDataset

cfg = OmegaConf.create(yaml.safe_load(open(f"{CKP}/config.yaml", "r")))

# traindata = CycloneDataset(
#     active_keys=cfg.dataset.active_keys,
#     path=cfg.dataset.path,
#     split="train",
#     random_seed=cfg.seed,
#     test_ratio=0.0,
#     normalization=cfg.dataset.normalization,
#     spatial_ifft=cfg.dataset.spatial_ifft,
#     in_memory=False,
#     input_seq_length=cfg.model.input_seq_length,
#     target_seq_length=cfg.model.bundle_seq_length,
#     trajectories=cfg.dataset.training_trajectories,
# )

data = CycloneDataset(
    active_keys=cfg.dataset.active_keys,
    path=cfg.dataset.path,
    split="val",
    random_seed=cfg.seed,
    test_ratio=0.0,
    normalization=None,
    spatial_ifft=False,
    in_memory=False,
    n_eval_steps=cfg.validation.n_eval_steps,
    input_seq_length=cfg.model.input_seq_length,
    target_seq_length=cfg.model.bundle_seq_length,
    trajectories=cfg.dataset.validation_trajectories,
)

print(f"Val: {len(data)}")

In [ ]:
model = get_model(cfg, dataset=data)

model, _, _ = load_model_and_config(f"{CKP}/ckp.pth", model, device)

model = model.to(device)

In [33]:
from einops import rearrange
import torch


def spatial_ifft(x):
    # invert fft on spatial
    x = rearrange(x, "b c ... -> c b ...")
    x = torch.complex(real=x[0], imag=x[1])
    x = torch.fft.ifftn(x, dim=(-2, -1))
    x = torch.stack([x.real, x.imag])
    x = rearrange(x, "c b ... -> b c ...")
    return x


def invert_ifft(x):
    x = rearrange(x, "b c ... -> c b ...")
    x = torch.complex(real=x[0], imag=x[1])
    x = torch.fft.fftn(x, dim=(-2, -1))
    x = torch.stack([x.real, x.imag])
    x = rearrange(x, "c b ... -> b c ...")
    return x


def normalize(x):
    # TODO proper normalization
    if cfg.dataset.normalization == "zscore":
        x_mean = x.mean((1, 2, 3, 4, 5), keepdims=True)
        x_std = x.std((1, 2, 3, 4, 5), keepdims=True)
        return (x - x_mean) / x_std, x_mean, x_std
    if cfg.dataset.normalization == "minmax":
        x_min = x.min((1, 2, 3, 4, 5), keepdims=True)
        x_max = x.max((1, 2, 3, 4, 5), keepdims=True)
        return (x - x_min) / (x_max - x_min), x_min, x_max

    return x, 0.0, 1.0


def denormalize(x, stat_1, stat_2):
    # TODO proper normalization
    if cfg.dataset.normalization == "zscore":
        return x * stat_2 + stat_1
    if cfg.dataset.normalization == "minmax":
        return x * (stat_2 - stat_1) + stat_1

    return x

In [52]:
OUT_DIR = "/system/user/publicdata/gyrokinetics/dumps/autoregressive_t0"
IDX_0 = 0
IDX_END = 25

In [53]:
losses = []
xt = data[IDX_0].x
xt = xt.to(device).unsqueeze(0)
xt = spatial_ifft(xt)
timesteps = data.timesteps
files = []

with torch.no_grad():
    for idx in range(IDX_0, IDX_END + 1):
        # xt = data[idx].x
        # xt = xt.to(device).unsqueeze(0)
        # xt = spatial_ifft(xt)
        ts = torch.tensor(timesteps[idx]).to(device).unsqueeze(0)
        xt, mean, std = normalize(xt)
        xt = model(xt, timestep=ts)
        xt = denormalize(xt, mean, std)

        b_xt = xt.clone()
        if cfg.dataset.spatial_ifft:
            b_xt = invert_ifft(b_xt)
        b_xt = b_xt.cpu().numpy().astype("float64").reshape(-1, order="F")
        # dump to file
        ftarget = os.path.join(OUT_DIR, f"K{str(int(idx)).zfill(5)}")
        with open(ftarget, "wb") as f:
            files.append(ftarget)
            print(f"Writing file {ftarget}")
            f.write(b_xt)

Writing file /system/user/publicdata/gyrokinetics/dumps/autoregressive_t0/K00000
Writing file /system/user/publicdata/gyrokinetics/dumps/autoregressive_t0/K00001
Writing file /system/user/publicdata/gyrokinetics/dumps/autoregressive_t0/K00002
Writing file /system/user/publicdata/gyrokinetics/dumps/autoregressive_t0/K00003
Writing file /system/user/publicdata/gyrokinetics/dumps/autoregressive_t0/K00004
Writing file /system/user/publicdata/gyrokinetics/dumps/autoregressive_t0/K00005
Writing file /system/user/publicdata/gyrokinetics/dumps/autoregressive_t0/K00006
Writing file /system/user/publicdata/gyrokinetics/dumps/autoregressive_t0/K00007
Writing file /system/user/publicdata/gyrokinetics/dumps/autoregressive_t0/K00008
Writing file /system/user/publicdata/gyrokinetics/dumps/autoregressive_t0/K00009
Writing file /system/user/publicdata/gyrokinetics/dumps/autoregressive_t0/K00010
Writing file /system/user/publicdata/gyrokinetics/dumps/autoregressive_t0/K00011
Writing file /system/user/pu

In [54]:
# files = [os.path.join(OUT_DIR, f) for f in os.listdir(OUT_DIR)]

In [55]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation, colormaps


def force_aspect(ax, aspect=1):
    im = ax.get_images()
    extent = im[0].get_extent()
    ax.set_aspect(abs((extent[1] - extent[0]) / (extent[3] - extent[2])) / aspect)


def plot4x4_animate(kfiles, dataset, title="", frames=5, start_idx=0):
    plt.rcParams["animation.html"] = "jshtml"
    plt.ioff()

    labels = ["vpar", "mu", "s", "x", "y"]
    comb = torch.combinations(torch.arange(5), 2).tolist()

    fig, ax = plt.subplots(5, 5, figsize=(30, 12))
    for i in range(5):
        for j in range(5):
            ax[i, j].remove()

    # fig.tight_layout()
    fig.suptitle(title)
    c_map = colormaps["coolwarm"]
    c_map.set_bad("k")

    preds = []
    for kfname in kfiles[:frames]:
        with open(kfname, "rb") as fid:
            kt = np.fromfile(fid, dtype=np.float64)
        kt = np.reshape(kt, (2, 32, 8, 16, 255, 32), order="F").astype("float32")
        preds.append(kt)

    def animate(t):
        sample = dataset[start_idx + t]
        x1 = sample.y.numpy()
        xp = preds[t]
        ts = sample.timestep.numpy().item()
        fig.suptitle(f"ts={ts:.2f}", fontsize=30)

        for i, j in comb:
            other = tuple([o for o in range(5) if o != i and o != j])

            x1_plot = x1[0].mean(other)
            xp_plot = xp[0].mean(other)
            x1_vmax, x1_vmin = x1_plot.max(), x1_plot.min()
            xp_vmax, xp_vmin = xp_plot.max(), xp_plot.min()

            # Clear the axis and directly plot two images side by side
            ax_ij = ax[i, j]
            # ax_ij.clear()

            # Get the position of the original axis
            pos = ax_ij.get_position()

            displ = pos.width / 2
            width = 0.92 * (displ)  # Split the width into two halves
            ax1 = fig.add_axes([pos.x0, pos.y0, width, pos.height])
            ax2 = fig.add_axes([pos.x0 + displ, pos.y0, width, pos.height])

            # Plot x1 and xp side by side
            ax1.imshow(x1_plot, cmap=c_map, vmax=x1_vmax, vmin=x1_vmin)
            ax2.imshow(xp_plot, cmap=c_map, vmax=xp_vmax, vmin=xp_vmin)

            if i == 0:
                # Set axis labels
                ax1.set_title("GT")
                ax2.set_title("PRED")
            ax1.set_xlabel(labels[i])
            ax1.set_ylabel(labels[j])
            ax2.set_xlabel(labels[i])
            ax2.set_ylabel(labels[j])

            # Force aspect ratio
            force_aspect(ax1)
            force_aspect(ax2)

    return animation.FuncAnimation(fig, animate, frames=frames)

In [57]:
ani = plot4x4_animate(files, dataset=data, frames=IDX_END - IDX_0, start_idx=IDX_0)
writer = animation.PillowWriter(fps=1, bitrate=600)
ani.save("autoreg.gif", writer=writer)